# IMI — Integrated Memory Intelligence
## Interactive Demo (5 minutes)

This notebook demonstrates IMI's cognitive memory capabilities:
1. **Encode** incidents as memories (with affect, affordances)
2. **Navigate** with adaptive relevance weighting
3. **Graph** multi-hop retrieval
4. **Dream** consolidation
5. **Compare** IMI vs pure cosine RAG

In [ ]:
from imi import IMISpace, Zoom

# Create a fresh memory space
space = IMISpace.from_sqlite('/tmp/imi_demo.db')
print(f'Memory space ready. Episodic: {len(space.episodic)}, Semantic: {len(space.semantic)}')

## 1. Encode Incidents
Each memory gets: embedding, affect (salience/valence), affordances (action suggestions), mass

In [ ]:
incidents = [
    ("DNS resolution failure at 03:00 UTC caused authentication service cascade failure across 3 microservices", ["dns", "auth", "incident"]),
    ("Auth service recovered after DNS fix at 03:45, but 200 users lost active sessions", ["auth", "recovery"]),
    ("Database connection pool exhausted during peak load, causing 500 errors on /api/users endpoint", ["database", "performance"]),
    ("Deployed circuit breaker pattern on auth service to prevent future cascade failures", ["auth", "improvement"]),
    ("SSL certificate for api.example.com expired, causing HTTPS handshake failures for mobile clients", ["ssl", "incident"]),
    ("Kubernetes pod OOMKilled: memory leak in image processing service consuming 4GB per pod", ["kubernetes", "memory"]),
    ("Rolling deployment of v2.3 caused 5-minute downtime due to backwards-incompatible API change", ["deployment", "incident"]),
    ("Redis sentinel failover triggered during network partition, 30 seconds of cache misses", ["redis", "incident"]),
    ("Implemented automated SSL cert renewal with Let's Encrypt, preventing future expiry incidents", ["ssl", "improvement"]),
    ("Post-mortem: DNS failure root cause was misconfigured TTL allowing stale records after provider migration", ["dns", "postmortem"]),
]

for text, tags in incidents:
    node = space.encode(text, tags=tags)
    affect = f'salience={node.affect.salience:.1f} valence={node.affect.valence:.1f}' if node.affect else 'none'
    affordances = len(node.affordances)
    print(f'  [{node.id[:8]}] mass={node.mass:.2f} affect=({affect}) affordances={affordances}')
    print(f'    {node.summary_medium[:80]}')
    print()

## 2. Navigate with Adaptive Relevance Weight
IMI detects query intent and adjusts the relevance weight automatically.

In [ ]:
queries = [
    "recent authentication failures",        # TEMPORAL → rw=0.15
    "find all SSL certificate incidents",     # EXPLORATORY → rw=0.00
    "how to prevent cascade failures",        # ACTION → rw=0.05
    "database performance issues",            # DEFAULT → rw=0.10
]

for q in queries:
    rw, intent = space.adaptive_rw.classify_with_info(q)
    result = space.navigate(q, top_k=3)
    print(f'\n🔍 "{q}"')
    print(f'   Intent: {intent.name} → rw={rw}')
    for m in result.memories[:3]:
        print(f'   [{m["score"]:.2f}] {m["content"][:80]}')

## 3. Affordance Search
Search by what actions memories ENABLE, not just content similarity.

In [ ]:
actions = space.search_affordances("prevent outages", top_k=5)
print(f'Actions for "prevent outages":\n')
for a in actions:
    print(f'  [{a["confidence"]:.0%}] {a["action"]}')
    print(f'       Conditions: {a["conditions"]}')
    print(f'       From: {a["memory_summary"][:60]}')
    print()

## 4. Graph: Multi-hop Retrieval
Add causal edges between related memories, then query across the chain.

In [ ]:
from imi.graph import EdgeType

# Get node IDs
nodes = space.episodic.nodes
dns_node = [n for n in nodes if 'DNS' in n.content and 'cascade' in n.content][0]
postmortem = [n for n in nodes if 'post-mortem' in n.content.lower() or 'postmortem' in n.content.lower()][0]
recovery = [n for n in nodes if 'recovered' in n.content.lower()][0]

# Create causal chain: DNS failure → recovery → postmortem
space.graph.add_edge(dns_node.id, recovery.id, EdgeType.CAUSAL, label='caused')
space.graph.add_edge(dns_node.id, postmortem.id, EdgeType.CAUSAL, label='investigated_in')

print(f'Graph: {space.graph.stats()}')
print(f'\nCausal chain from DNS incident:')
for nid, weight in space.graph.neighbors(dns_node.id):
    neighbor = next((n for n in nodes if n.id == nid), None)
    if neighbor:
        print(f'  → {neighbor.summary_medium[:80]}')

In [ ]:
# Now navigate — graph expansion finds the full chain
result_no_graph = space.navigate("what caused the auth cascade?", use_graph=False, top_k=3)
result_with_graph = space.navigate("what caused the auth cascade?", use_graph=True, top_k=3)

print('Without graph expansion:')
for m in result_no_graph.memories[:3]:
    print(f'  [{m["score"]:.2f}] {m["content"][:80]}')

print('\nWith graph expansion:')
for m in result_with_graph.memories[:3]:
    print(f'  [{m["score"]:.2f}] {m["content"][:80]}')

## 5. Dream (Consolidation)
Cluster similar memories into semantic patterns — like sleep consolidation.

In [ ]:
report = space.dream()
print(f'Dream cycle complete:')
print(f'  Nodes processed: {report.nodes_processed}')
print(f'  Clusters formed: {report.clusters_formed}')
print(f'  Patterns: {report.patterns_total}')
print(f'  Semantic store: {len(space.semantic)} consolidated memories')

## 6. Zoom Levels
Same query, different resolution levels.

In [ ]:
for zoom in [Zoom.ORBITAL, Zoom.MEDIUM, Zoom.DETAILED]:
    result = space.navigate("DNS incident", zoom=zoom, top_k=1)
    if result.memories:
        m = result.memories[0]
        print(f'\n{zoom.value.upper()}:')
        print(f'  {m["content"][:120]}')

## 7. IMI vs Pure Cosine
The key insight: cognitive features trade retrieval accuracy for agent relevance.

In [ ]:
import numpy as np

query = "what should I do about the DNS issue?"
query_emb = space.embedder.embed(query)

# Pure cosine (rw=0)
pure_cosine = space.navigate(query, relevance_weight=0.0, top_k=3)
# IMI adaptive (auto rw)
imi_adaptive = space.navigate(query, top_k=3)

rw, intent = space.adaptive_rw.classify_with_info(query)

print(f'Query: "{query}"')
print(f'Detected intent: {intent.name} → rw={rw}')
print(f'\nPure cosine (rw=0.0):')
for m in pure_cosine.memories[:3]:
    print(f'  [{m["score"]:.3f}] {m["content"][:70]}')

print(f'\nIMI adaptive (rw={rw}):')
for m in imi_adaptive.memories[:3]:
    print(f'  [{m["score"]:.3f}] {m["content"][:70]}')

print(f'\n--- Summary ---')
print(f'Cognitive features change ranking to favor actionable, recent memories.')
print(f'Trade-off: -3.7% R@5 for +6.7% agent precision, +100% multi-hop recall.')

## Summary

| Feature | What it does |
|---------|-------------|
| **Adaptive RW** | Auto-detects query intent, adjusts relevance weight |
| **Affordances** | "What can I DO?" not just "What do I know?" |
| **Graph expansion** | Multi-hop causal chain retrieval |
| **Dream** | Consolidates similar memories into patterns |
| **Zoom** | Same query, different resolution levels |
| **Affect** | Critical incidents resist forgetting |

**Zero LLM calls at query time. SQLite only. 84 tests.**

Learn more: [GitHub](https://github.com/RAG7782/imi) | [Paper](../docs/arxiv/imi-paper.pdf)